# Case study 1 — Novel Baswedan tweet corpus (Indonesian)

Preprocessing walk-through and exploratory analysis of 2,727 Indonesian tweets
about the Novel Baswedan acid-attack case.

This corpus has **no sentiment label**, so nothing here trains a classifier —
it cleans the data, quantifies the original pipeline's mention bug on real rows,
and describes the corpus (hashtags, terms, entity clusters).

Executed end-to-end; every output below is real.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)


repo root: /home/heisenberg/portfolio-rework/repos/indonesian-tweet-sentiment-analysis


In [2]:
import re
import pandas as pd
from idtweets.cases import load_case_module
from idtweets.preprocessing import remove_pattern_original, strip_mentions, MENTION_RE

case = load_case_module('novel-baswedan', 'novel_baswedan_analysis.py')
raw = case.load_raw()
print(raw.shape)
raw.head(3)

(2727, 2)


,id_str,text
0,1274657459018149888,"RT @amnestyindo: [BREAKING] ""Dengan rekam jeja..."
1,1274657246996135937,RT @Aryprasetyo85: #RekayasaDidzolimi \nEkseku...
2,1274657226234359808,RT @AriestaRiico_: Jika Novel berteriak minta ...


## Defect #1 — the original `remove_pattern` kept all but the first mention

The original helper returned from inside its `for` loop. Let us run it on a real
tweet from this corpus that mentions four accounts.

In [3]:
ex = raw[raw['text'].astype(str).str.count(r'@\w+') >= 4]['text'].iloc[0]
print('RAW:')
print(ex[:180], '\n')
orig = remove_pattern_original(ex, r'@\w+')
fixed = strip_mentions(ex)
print('original logic -> mentions still present:', re.findall(r'@\w+', orig))
print('fixed logic    -> mentions still present:', re.findall(r'@\w+', fixed))

RAW:
RT @dikisahkeun: @kepersekutuanm @brizcky98 @dapppoy @txtdrberseragam Ngelacak begituan aja cepet, semangat 45, kasus kasus dari zaman orba, sampe kasus novel baswedan aja lamaaa b 

original logic -> mentions still present: ['@kepersekutuanm', '@brizcky98', '@dapppoy', '@txtdrberseragam']
fixed logic    -> mentions still present: []


In [4]:
impact = case.defect_impact(raw)
pd.Series(impact, name='value').to_frame()

,value
rows_total,2727.000000
rows_with_any_mention,2510.000000
rows_with_multiple_mentions,634.000000
mentions_total,3616.000000
original_stripped_total,2517.000000
original_left_behind_total,1099.000000
fixed_left_behind_total,0.000000
rows_original_got_wrong,627.000000
retweet_share,0.867620
duplicate_texts,2136.000000


Across the corpus the original left **1,099 of 3,616 mentions** in place across
**627 tweets**; the corrected function leaves zero.

## Preprocessing and de-duplication

In [5]:
processed = case.preprocess(raw)
deduped = case.deduplicate(processed)
comp = pd.DataFrame({
    'raw': raw['text'].astype(str).head(4),
    'clean': processed['clean'].head(4),
    'tokens': processed['clean_token_count'].head(4),
})
pd.set_option('display.max_colwidth', 90)
comp

,raw,clean,tokens
0,"RT @amnestyindo: [BREAKING] ""Dengan rekam jejak komitmen negara saat ini, kita perlu m...",breaking rekam jejak komitmen negara waris generasi adil ya negara bukti komitmen tunt...,15
1,RT @Aryprasetyo85: #RekayasaDidzolimi \nEksekusi penembakan dilakukan pada malam hari....,rekayasadidzolimi eksekusi tembak malam novel tembak kaki kiri erwansyah senjata api j...,24
2,"RT @AriestaRiico_: Jika Novel berteriak minta keadilan, bagaimana dengan keadilan korb...",novel teriak adil adil korban laku sarang burung walet tembak adil hak warga negara re...,15
3,"RT @TeddyGusnaidi: Novel jadi korban penyiraman air keras, wajar jika menuntut keadila...",novel korban siram air keras wajar tuntut adil korban aniaya sarang burung wallet waja...,24


In [6]:
print(f'raw rows                  : {len(raw)}')
print(f'rows after preprocessing  : {len(processed)}')
print(f'rows after de-duplication : {len(deduped)}')
print(f'duplicate share           : {raw["text"].astype(str).duplicated().mean():.1%}')
print(f'empty cleaned rows        : {(processed["clean"].str.len() == 0).sum()}')

raw rows                  : 2727
rows after preprocessing  : 2727
rows after de-duplication : 462
duplicate share           : 78.3%
empty cleaned rows        : 18


## What the corpus is about

In [7]:
tags = case.hashtag_table(raw, top_n=12)
tags

,hashtag,count
0,RekayasaDidzolimi,1366
1,BasmiCovidDanKadrunKomunis,46
2,EnggakSengaja,39
3,PKSPenyelamatPancasila,35
4,NovelBaswedan,22
5,HBD59Jokowi,14
6,GakSengaja,13
7,Indonesia,10
8,Viral,9
9,menolaklupa,9


In [8]:
case.entity_table(deduped)

,entity,matching_tweets,share
0,Novel Baswedan,419,0.9069
1,Acid attack (air keras),82,0.1775
2,Jokowi,71,0.1537
3,KPK,29,0.0628
4,Prosecutors / JPU,28,0.0606
5,Corruption,28,0.0606
6,Police / Bareskrim,21,0.0455
7,Human rights,8,0.0173
8,Ombudsman,3,0.0065
9,Rizieq,1,0.0022


In [9]:
case.top_terms(deduped, top_n=20, min_df=3)

,term,count
0,novel,473
1,baswedan,431
2,hukum,95
3,air,92
4,keras,84
5,siram,82
6,adil,79
7,rekayasadidzolimi,72
8,jokowi,61
9,tuntut,59


## Takeaways

* Cleaning is not cosmetic here: 78% of the rows are duplicate retweets and
  87% are retweets, so the de-duplicated corpus (462 tweets) is what a reader
  should reason about.
* The hashtag distribution is dominated by a single campaign tag
  (`#RekayasaDidzolimi`), which is a strong hint that this dump is a
  coordinated capture rather than a random sample of Indonesian opinion —
  another reason not to read sentiment off it.
* The original pipeline's mention bug is not theoretical: it silently kept
  ~30% of all mentions, which contaminates any downstream vocabulary or
  feature built from this text.